# 02 Data Cleaning

> Change only `RAW` / `PROCESSED` paths if your folder location is different.

## 1. Paths and imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re, json

BASE = Path(r"C:\CHANGE\THIS\TO\YOUR\PROJECT")
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
arrivals = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_mandi_arrivals.csv")
master = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_mandi_master.csv")
transport = pd.read_csv(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_transport_logistics.csv")
master
with open(RAW / "C:\\Users\\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_price_and_msp.json", encoding="utf-8") as f:
    prices = pd.DataFrame(json.load(f))

weather = pd.read_excel(RAW / "C:\\Users\yalamanchi rohitha\\Downloads\\track3_agritech_dataset_files\\track3_weather_sensors.xlsx")

print(arrivals.shape, master.shape, prices.shape, weather.shape, transport.shape)

(25750, 8) (60, 6) (12000, 9) (15000, 7) (10400, 10)


## 2. Standardize column names

In [2]:
def clean_columns(df):
    df = df.copy()
    df.columns = (df.columns.astype(str).str.strip().str.lower()
                  .str.replace(r"[^a-z0-9]+", "_", regex=True)
                  .str.strip("_"))
    return df

arrivals, master, prices, weather, transport = map(
    clean_columns, [arrivals, master, prices, weather, transport]
)

## 3. Clean crop names

In [3]:
# ============================================
# FINAL STANDARDIZATION CHECK
# ============================================

import re
import pandas as pd
import numpy as np

# ---------- Crop standardization ----------
crop_map = {
    "basmati": "Rice",
    "chawal": "Rice",
    "rice": "Rice",
    "paddy": "Rice",
    "धान": "Rice",
    "चावल": "Rice",
    
    "corn": "Maize",
    "maize": "Maize",
    "makka": "Maize",
    "makki": "Maize",
    "मक्का": "Maize",
    
    "cotton": "Cotton",
    "kapas": "Cotton",
    "narma": "Cotton",
    "कपास": "Cotton",
    
    "wheat": "Wheat",
    "gehun": "Wheat",
    "gehu": "Wheat",
    "kanak": "Wheat",
    "गेहूं": "Wheat",
    
    "mustard": "Mustard",
    "sarso": "Mustard",
    "sarson": "Mustard",
    "सरसों": "Mustard",
    
    "sugarcane": "Sugarcane",
    "ganna": "Sugarcane",
    "ganne": "Sugarcane",
    "गन्ना": "Sugarcane"
}

def normalize_crop(x):
    if pd.isna(x):
        return np.nan
    
    s = str(x).strip().lower()
    return crop_map.get(s, str(x).strip().title())

for df in [arrivals, prices]:
    if "crop_name" in df.columns:
        df["crop_name"] = df["crop_name"].apply(normalize_crop)


# ---------- Mandi ID standardization ----------
def normalize_mandi_id(x):
    if pd.isna(x):
        return np.nan
    
    s = str(x).strip().upper()
    
    # Extract numeric part from values such as:
    # MANDI019, mandi_019, Mandi019
    match = re.search(r"(\d+)", s)
    
    if match:
        return f"MANDI{int(match.group(1)):03d}"
    
    return s

for df in [arrivals, prices, master, transport]:
    if "mandi_id" in df.columns:
        df["mandi_id"] = df["mandi_id"].apply(normalize_mandi_id)


# ---------- Show final standardization ----------
print("Final crop values in arrivals:")
print(sorted(arrivals["crop_name"].dropna().unique()))

print("\nFinal mandi ID examples:")
print(arrivals["mandi_id"].dropna().head(20).tolist())

print("\nUnique crop count:", arrivals["crop_name"].nunique())
print("Unique mandi ID count:", arrivals["mandi_id"].nunique())

Final crop values in arrivals:
['Cotton', 'Dhaan', 'Maize', 'Mustard', 'Rice', 'Sugarcane', 'Wheat']

Final mandi ID examples:
['MANDI026', 'MANDI014', 'MANDI056', 'MANDI019', 'MANDI050', 'MANDI049', 'MANDI054', 'MANDI045', 'MANDI029', 'MANDI010', 'MANDI044', 'MANDI042', 'MANDI056', 'MANDI011', 'MANDI025', 'MANDI050', 'MANDI006', 'MANDI039', 'MANDI001', 'MANDI028']

Unique crop count: 7
Unique mandi ID count: 57


## 4. Clean mandi IDs and dates

In [4]:
for df in [arrivals, prices, master, transport]:
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

if "arrival_time" in transport.columns:
    transport["arrival_time"] = pd.to_datetime(
        transport["arrival_time"],
        errors="coerce"
    )

## 5. Clean arrivals quantity

In [5]:
# ==========================================
# 5. CLEAN ARRIVAL QUANTITY
# ==========================================

# Convert arrival quantity from messy text to number
def to_number(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    # Remove common currency/number symbols
    s = s.replace(",", "")
    s = s.replace("₹", "")
    s = s.replace("Rs.", "")
    s = s.replace("Rs", "")
    s = s.replace("INR", "")

    # Keep only numbers, decimal point and minus sign
    s = re.sub(r"[^0-9.\-]", "", s)

    return pd.to_numeric(s, errors="coerce")


# Clean quantity
arrivals["arrival_quantity"] = arrivals["arrival_quantity"].apply(to_number)


# Clean unit names
arrivals["unit"] = (
    arrivals["unit"]
    .astype("string")
    .str.strip()
    .str.lower()
)


# Conversion factors to Quintals (Qtl)
# 1 Qtl = 100 KG
# 1 Tonne = 10 Qtl

unit_map = {
    "kg": 0.01,
    "kgs": 0.01,
    "kilo": 0.01,

    "q": 1,
    "qtl": 1,
    "quintal": 1,
    "quintals": 1,

    "mt": 10,
    "t": 10,
    "tonne": 10,
    "tonnes": 10
}


# Create conversion factor
arrivals["quantity_to_qtl_factor"] = arrivals["unit"].map(unit_map)


# Convert everything to Quintals
arrivals["arrival_quantity_qtl"] = (
    arrivals["arrival_quantity"]
    * arrivals["quantity_to_qtl_factor"]
)


# Display result
display(
    arrivals[
        [
            "arrival_quantity",
            "unit",
            "quantity_to_qtl_factor",
            "arrival_quantity_qtl"
        ]
    ].head(10)
)


print(
    "Missing converted quantities:",
    arrivals["arrival_quantity_qtl"].isna().sum()
)

,arrival_quantity,unit,quantity_to_qtl_factor,arrival_quantity_qtl
0,44.841,t,10.00,448.4100
1,332.540,qtl,1.00,332.5400
2,415.880,<NA>,NaN,NaN
3,-107.130,kg,0.01,-1.0713
4,-331.590,qtl,1.00,-331.5900
5,119.930,q,1.00,119.9300
6,187.680,qtl,1.00,187.6800
7,393.670,<NA>,NaN,NaN
8,14.840,quintals,1.00,14.8400
9,93.280,qtl,1.00,93.2800


Missing converted quantities: 5143


## 6. Clean prices and MSP

In [6]:
for c in ["min_price", "max_price", "modal_price", "msp"]:
    if c in prices.columns:
        prices[c] = prices[c].apply(to_number)

prices["district"] = prices["district"].astype("string").str.strip().str.title()
master["district"] = master["district"].astype("string").str.strip().str.title()
master["state"] = master["state"].astype("string").str.strip().str.title()
master["mandi_type"] = master["mandi_type"].astype("string").str.strip().str.title()
master = master.drop_duplicates().copy()

## 7. Clean weather

In [7]:
weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="coerce", dayfirst=True, utc=True)

weather["temperature"] = weather["temperature"].apply(to_number)
weather["temp_unit"] = weather["temp_unit"].astype("string").str.strip().str.lower()

weather.loc[weather["temp_unit"].isin(["f","°f","fahrenheit"]), "temperature"] = (
    weather.loc[weather["temp_unit"].isin(["f","°f","fahrenheit"]), "temperature"] - 32
) * 5/9

weather["temp_c"] = weather["temperature"]
weather["rainfall"] = weather["rainfall"].apply(to_number)
weather["rain_unit"] = weather["rain_unit"].astype("string").str.strip().str.lower()

inch_mask = weather["rain_unit"].isin(["in","inch","inches"])
weather.loc[inch_mask, "rainfall_mm"] = weather.loc[inch_mask, "rainfall"] * 25.4
weather.loc[~inch_mask, "rainfall_mm"] = weather.loc[~inch_mask, "rainfall"]

weather["humidity_percent"] = pd.to_numeric(weather["humidity_percent"], errors="coerce")
weather["date"] = weather["timestamp"].dt.tz_convert("Asia/Kolkata").dt.date

## 8. Clean transport

In [8]:
transport["transit_hours"] = pd.to_numeric(transport["transit_hours"], errors="coerce")
transport.loc[transport["transit_hours"] < 0, "transit_hours"] = np.nan

transport["distance"] = transport["distance"].apply(to_number)
transport["distance_unit"] = transport["distance_unit"].astype("string").str.strip().str.lower()
mile_mask = transport["distance_unit"].isin(["mile","miles","mi"])
transport["distance_km"] = transport["distance"]
transport.loc[mile_mask, "distance_km"] = transport.loc[mile_mask, "distance"] * 1.60934

transport["vehicle_no"] = (transport["vehicle_no"].astype("string")
                           .str.upper().str.replace(r"[^A-Z0-9]", "", regex=True))

## 9. Remove exact duplicates and create quality flags

In [9]:
raw_counts = {
    "arrivals_raw": len(arrivals), "prices_raw": len(prices),
    "weather_raw": len(weather), "transport_raw": len(transport), "master_raw": len(master)
}

arrivals = arrivals.drop_duplicates().copy()
prices = prices.drop_duplicates().copy()
weather = weather.drop_duplicates().copy()
transport = transport.drop_duplicates().copy()

arrivals["quality_missing_quantity"] = arrivals["arrival_quantity_qtl"].isna()
arrivals["quality_missing_mandi"] = arrivals["mandi_id"].isna()
prices["quality_missing_price"] = prices["modal_price"].isna()
weather["quality_missing_weather"] = weather["temp_c"].isna() & weather["rainfall_mm"].isna()
transport["quality_invalid_transit"] = transport["transit_hours"].isna()

print("Raw vs cleaned:")
print(raw_counts)
print({"arrivals_clean":len(arrivals),"prices_clean":len(prices),
       "weather_clean":len(weather),"transport_clean":len(transport),"master_clean":len(master)})

Raw vs cleaned:
{'arrivals_raw': 25750, 'prices_raw': 12000, 'weather_raw': 15000, 'transport_raw': 10400, 'master_raw': 57}
{'arrivals_clean': 25000, 'prices_clean': 12000, 'weather_clean': 15000, 'transport_clean': 10000, 'master_clean': 57}


## 10. Save cleaned datasets

In [10]:
arrivals.to_csv(PROCESSED / "cleaned_arrivals.csv", index=False)
prices.to_csv(PROCESSED / "cleaned_prices.csv", index=False)
weather.to_csv(PROCESSED / "cleaned_weather.csv", index=False)
transport.to_csv(PROCESSED / "cleaned_transport.csv", index=False)
master.to_csv(PROCESSED / "cleaned_master.csv", index=False)

print("Cleaning complete.")

Cleaning complete.
